# Cisco Skill Scanner on GitSkills — RQ1 experiment notebook

**MSR 2027 Mining Challenge · SkillGuard team · RQ1 (how skill-package context affects security findings)**

This notebook takes agent skills from the GitSkills SQLite database, rebuilds each skill as files on disk in
two versions (the `SKILL.md` instruction file alone, and the full skill folder with its supporting files), runs the
Cisco AI Defense Skill Scanner over both versions with two analyzer settings, and turns the scanner's JSON
reports into tables and charts that address our research questions:

| Question | What we compare |
|---|---|
| **RQ1-A Context and detection** | the same skill scanned as `SKILL.md` only vs. as its full folder |
| **RQ1-B Security behavior coverage** | which Cisco rules and categories fire, and how they map to our research categories |
| **RQ1-C Limitations and relevance** | what Cisco does not report, how much the analyzer configuration matters, and how many scans were usable |

### Vocabulary we keep separate

A scanner finding is **evidence, not a verdict**. Throughout the notebook we distinguish:

1. an **observed operation** (the code really does send an HTTP request),
2. a **scanner warning** (Cisco matched a rule on that operation),
3. a **potential vulnerability** (a human judged that the operation could be abused), and
4. **confirmed malicious behavior** (a human confirmed intent).

The notebook only produces items 1 and 2. Items 3 and 4 need a person; Section 13 writes a manual-review
template for that.

### Safety

* The database is opened **read-only**. Nothing here writes to it.
* Skill contents are **untrusted**. The notebook writes them to disk and hands the folder to the scanner; it
  **never executes** a skill's scripts, and the scanner does not execute them either (it uses pattern, YARA,
  AST-dataflow and bytecode analysis).
* Supporting-file paths from the database are checked before writing so nothing can escape the export folder.
* Exported skills and reports live under `results/cisco_dataset_runs/`; do not commit them without team agreement.

### How to run it

1. Select the project `.venv` kernel in VS Code.
2. Change the values in **Section 1 (Settings)**. Nothing else needs editing for a normal run.
3. **Run All**. Every section depends only on the sections above it.

Section map: **1** settings · **2** imports · **3** database · **4** scanner and configurations ·
**5** select skills · **6** export skill folders · **7** run Cisco · **8** flatten findings · **9** save ·
**10–13** analysis (A security findings, B context, C analyzers, D reliability) · **14** interpretation and scaling.

## 1 — Settings

Everything a researcher normally changes is in the next cell. Each setting has a comment with example values.

* **Database.** `"sample"` uses the official 13k-skill sample. For the full release give a path relative to the
  project root (for example `"data/full/agent_skills.db"`) or an absolute path.
* **Selection mode.** `pilot` scans one of our known pilot skills; `exact` scans one repo/path you type;
  `search` only *previews* skills whose `SKILL.md` contains a keyword (no scanning); `batch` scans the first
  `MAX_SKILLS` keyword matches. Keyword search is a convenience for exploring, not our sampling method; the
  stratified random sample for the 400-skill study is a separate step (Section 14).
* **Scanner.** `RUN_SCANS = False` re-reads reports saved by an earlier run instead of calling Cisco, so the
  analysis sections can be re-run without re-scanning. `POLICY` is Cisco's scan policy preset; `balanced`
  is Cisco's default and is what our pilot results were produced with. Keep it fixed within a study.
* **Optional analyzers** are off by default. `ENABLE_OSV` queries the OSV.dev vulnerability database over the
  network (no key). `ENABLE_LLM_CONFIGS` adds two LLM-judge configurations; it needs an API key in the
  environment variable `SKILL_SCANNER_LLM_API_KEY`, costs money per scan, and **sends skill contents to the
  model provider**. Do not enable it without team agreement.

In [ ]:
# ---------------------------------------------------------------- Database --
DATABASE = "sample"          # "sample"  or a path such as "data/full/agent_skills.db"
HASH_DATABASE = True         # record the SHA-256 of the database file (takes ~1 s for the sample; set False for a 44 GB file)

# --------------------------------------------------------- Skill selection --
MODE = "pilot"               # "pilot" | "exact" | "search" | "batch"
PILOT = "canary-tokens"      # one of the PILOTS keys below
EXACT_REPO = "AceDataCloud/Skills"      # used when MODE == "exact"
EXACT_PATH = "skills/x/SKILL.md"
SEARCH_KEYWORD = "API_KEY"   # used when MODE is "search" or "batch"; case-insensitive text in SKILL.md
PREVIEW_LIMIT = 15           # rows shown in search mode (1..100)
MAX_SKILLS = 3               # skills scanned in batch mode (1..20); larger studies run as several documented batches

PILOTS = {
    # label: (repository, path of SKILL.md inside the repository)
    "loa-credentials": ("0xHoneyJar/loa-dixie", ".claude/skills/managing-credentials/SKILL.md"),
    "canary-tokens": ("26zl/cybersec-toolkit", ".claude/skills/implementing-canary-tokens-for-network-intrusion/SKILL.md"),
    "x": ("AceDataCloud/Skills", "skills/x/SKILL.md"),
}

# ------------------------------------------------------------------ Scanner --
RUN_SCANS = True             # False: do not call Cisco, load reports saved by an earlier run
POLICY = "balanced"          # Cisco preset: "balanced" (default) | "strict" | "permissive" | "low-noise" | "quiet"
LENIENT = False              # True adds --lenient (tolerate malformed SKILL.md frontmatter); keep False for comparability
SCAN_TIMEOUT_SECONDS = 300   # one scanner call is normally 5-10 s; this only stops a hung process
ALLOW_INCOMPLETE_FOLDERS = False  # True scans "full" folders even when GitSkills did not store every file (reported as incomplete)

# -------------------------------------------------------- Optional analyzers --
ENABLE_OSV = False           # add --use-osv to every configuration (dependency vulnerabilities; network, no key)
ENABLE_LLM_CONFIGS = False   # add two "LLM judge" configurations; needs SKILL_SCANNER_LLM_API_KEY, paid, sends content to a provider
LLM_PROVIDER = "anthropic"   # "anthropic" | "openai" | "openai-compatible" (only used when ENABLE_LLM_CONFIGS is True)

# Basic sanity checks so a typo fails here, not three sections later.
assert MODE in ("pilot", "exact", "search", "batch"), f"Unknown MODE: {MODE!r}"
assert PILOT in PILOTS, f"Unknown PILOT: {PILOT!r}; choose from {sorted(PILOTS)}"
assert 1 <= PREVIEW_LIMIT <= 100
assert 1 <= MAX_SKILLS <= 20, "Batch cap is 20 skills per run; split larger studies into documented batches"
assert POLICY in ("balanced", "strict", "permissive", "low-noise", "quiet"), f"Unknown POLICY: {POLICY!r}"
print("Settings loaded. MODE =", MODE, "| RUN_SCANS =", RUN_SCANS, "| POLICY =", POLICY)

## 2 — Imports and project folders

Standard-library modules we rely on:

* `sqlite3` talks to the GitSkills database; `pandas` turns query results into DataFrames (tables).
* `pathlib.Path` represents file paths. `ROOT / "data" / "sample"` builds a path piece by piece, and
  `.resolve()` turns it into an absolute path with symlinks and `..` removed.
* `hashlib` computes SHA-256 hashes so we can prove which bytes were scanned.
* `subprocess` launches the Cisco command-line tool as a separate process (Section 7).
* `json` reads Cisco's reports; `time` measures how long each scan takes.

The project root is found by walking up from the notebook's folder until a folder with both `notebooks/` and
`data/` is found, so the notebook works whether VS Code starts it in `notebooks/` or in the repository root.

In [ ]:
import hashlib
import json
import os
import re
import shutil
import sqlite3
import subprocess
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 80)   # characters shown per table cell; raise to read longer text
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

cwd = Path.cwd().resolve()
ROOT = next((folder for folder in (cwd, *cwd.parents) if (folder / "notebooks").is_dir() and (folder / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the project repository (it needs the notebooks/ and data/ folders).")

RESULTS_ROOT = ROOT / "results" / "cisco_dataset_runs"
print("Project root :", ROOT)
print("Results root :", RESULTS_ROOT)

## 3 — Connect to the GitSkills database (read-only)

GitSkills stores every `SKILL.md` found on GitHub in the `artifacts` table and the other files of each skill
folder in `artifact_siblings`. Two schema facts matter for this experiment:

* **Copies share text.** The same `SKILL.md` is often vendored into many repositories. Copies share a
  `file_sha`; only one *representative* row per `file_sha` (`dedup_primary = 1`) stores the text and the
  supporting files. If you ask for a copy, Section 5 resolves it to its representative.
* **Not every supporting file has text.** `artifact_siblings.content_fetched` is `1` when the file text is
  stored and `2` when it was skipped (binary, oversized, or the folder had too many files). A folder with any
  `content_fetched != 1` file cannot be rebuilt completely; Section 6 records that and Section 7 skips the
  full-folder scan unless `ALLOW_INCOMPLETE_FOLDERS` is set.

The connection uses SQLite's `mode=ro` URI flag plus `PRAGMA query_only`, so an accidental `UPDATE` would fail
instead of changing the dataset. `sql("SELECT ...", (params,))` is the only way the notebook reads the
database; `?` placeholders keep values out of the SQL text.

**Output:** connection details, the database hash (for the reproducibility record), and row counts.

In [ ]:
if DATABASE == "sample":
    DB_PATH = ROOT / "data" / "sample" / "agent_skills_sample.db"
else:
    DB_PATH = Path(DATABASE).expanduser()
    if not DB_PATH.is_absolute():
        DB_PATH = ROOT / DB_PATH
DB_PATH = DB_PATH.resolve()

if not DB_PATH.is_file():
    local_dbs = sorted({*ROOT.glob("data/**/*.db"), *ROOT.glob("data/**/*.sqlite"), *ROOT.glob("data/**/*.sqlite3")})
    found = "\n".join(f"  - {p.relative_to(ROOT)} ({p.stat().st_size:,} bytes)" for p in local_dbs) or "  (none)"
    raise FileNotFoundError(f"Database not found: {DB_PATH}\nDatabases found under data/:\n{found}\nSet DATABASE in Section 1.")

# Read-only connection: the URI flag refuses writes at the file level, the PRAGMA refuses them at the SQL level.
conn = sqlite3.connect(DB_PATH.as_uri() + "?mode=ro", uri=True)
conn.execute("PRAGMA query_only = ON")
conn.execute("PRAGMA busy_timeout = 15000")


def sql(query, params=()):
    """Run a read-only SQL query and return the rows as a pandas DataFrame."""
    return pd.read_sql_query(query, conn, params=params)


def table_columns(table):
    """Return the set of column names of a table (PRAGMA table_info lists one row per column)."""
    return {row[1] for row in conn.execute(f'PRAGMA table_info("{table}")')}


def sha256_of_file(path, chunk_bytes=8 * 1024 * 1024):
    """SHA-256 of a file, read in chunks so very large files do not need to fit in memory."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


# The columns the rest of the notebook relies on. A database with a different schema needs an adapter first.
REQUIRED_COLUMNS = {
    "artifacts": {"repo_full_name", "path", "filename", "content", "file_sha", "dedup_primary", "name",
                  "frontmatter_valid", "sibling_count", "has_scripts", "has_references",
                  "composition_fetched", "composition_truncated"},
    "artifact_siblings": {"repo_full_name", "artifact_path", "entry_name", "entry_type", "content", "content_fetched"},
}
for table, needed in REQUIRED_COLUMNS.items():
    missing = needed - table_columns(table)
    if missing:
        raise ValueError(f"Incompatible database schema: table {table!r} lacks columns {sorted(missing)}")

DB_TAG = re.sub(r"[^a-zA-Z0-9_-]", "_", DB_PATH.stem)[:45]   # short safe name used in output folder names
DB_SHA256 = sha256_of_file(DB_PATH) if HASH_DATABASE else None

print("Connected (read-only):", DB_PATH)
print("Database tag         :", DB_TAG)
print("Database SHA-256     :", DB_SHA256 or "(not computed)")
print("Integrity quick_check:", conn.execute("PRAGMA quick_check").fetchone()[0])
display(sql("""
    SELECT COUNT(*)                        AS skill_md_occurrences,
           SUM(dedup_primary)              AS distinct_skill_files,
           SUM(dedup_primary = 1 AND sibling_count > 0) AS distinct_with_supporting_files,
           COUNT(DISTINCT repo_full_name)  AS repositories
    FROM artifacts
"""))

## 4 — The Cisco scanner and our four configurations

[Cisco AI Defense Skill Scanner](https://github.com/cisco-ai-defense/skill-scanner) (PyPI package
`cisco-ai-skill-scanner`, command `skill-scanner`) scans a **skill folder** and reports *findings*. Each
finding has a rule ID, a Cisco threat category, a severity (`CRITICAL`, `HIGH`, `MEDIUM`, `LOW`, `INFO`), the
analyzer that produced it, and where possible a file path, line number, source snippet and remediation text.
The overall verdict `is_safe` is simply "no `CRITICAL` or `HIGH` finding" — a skill can be `is_safe = True`
with several `MEDIUM` findings, so we never use the verdict alone.

What the analyzers do (as installed, version printed below):

| Analyzer | Enabled by | What it looks at |
|---|---|---|
| `static` | always | ~47 YAML signature rules (regexes per file type), 27 YARA rules, and Python checks on the manifest, hidden files, binaries, file-type mismatches, undeclared network use |
| `bytecode` | always | `.pyc` files that do not match their source |
| `pipeline` | always | shell command chains (`curl … \| sh`, taint through pipes) |
| `correlation` | always (policy) | bounded source-to-sink flows across files (e.g. sensitive read → network send) |
| `behavioral` | `--use-behavioral` | Python AST dataflow: env-var harvesting plus network, eval/exec with subprocess, cross-file flows; also code blocks inside Markdown (`MDBLOCK_*` rules) |
| `llm` | `--use-llm` | an LLM judge reads `SKILL.md` and scripts for semantic concerns; needs an API key; paid; content leaves the machine |
| `osv` | `--use-osv` | OSV.dev lookup of exactly pinned Python/JS dependencies; network, no key |
| `trigger`, `meta`, `virustotal`, `aidefense` | flags | description specificity; LLM false-positive filter (Cisco says it costs recall); hash reputation; Cisco cloud classifier |

The **scan policy** (`--policy`) controls thresholds, allowlists, rule severity overrides and which core analyzers
run; `balanced` is Cisco's default. We pass it explicitly so every report records the same policy fingerprint.

Our experiment is a 2 × 2 design, built below as `SCAN_CONFIGS`:

| | default analyzers | `--use-behavioral` |
|---|---|---|
| **SKILL.md only** | configuration 1 | configuration 2 |
| **Full folder** | configuration 3 | configuration 4 |

Configurations 1 vs 3 and 2 vs 4 answer RQ1-A (context); 1 vs 2 and 3 vs 4 answer the analyzer part of RQ1-C.
Optional LLM configurations (5, 6) are appended only when `ENABLE_LLM_CONFIGS` is on and a key is present.

**Output:** the scanner path and version, and a table of the configurations that will run.

In [ ]:
scanner_candidates = [ROOT / ".venv/bin/skill-scanner", ROOT / ".venv/Scripts/skill-scanner.exe"]
SCANNER = next((str(p) for p in scanner_candidates if p.is_file()), None) or shutil.which("skill-scanner")

if SCANNER is None:
    if RUN_SCANS:
        raise FileNotFoundError(
            "Cisco skill-scanner not found. Install it into the project environment with\n"
            "    .venv/bin/pip install 'cisco-ai-skill-scanner==2.3.1'\n"
            "(pin the version the team agreed on), or set RUN_SCANS = False to only read saved reports."
        )
    SCANNER_VERSION = "not installed"
else:
    version_proc = subprocess.run([SCANNER, "--version"], capture_output=True, text=True, timeout=60)
    SCANNER_VERSION = (version_proc.stdout or version_proc.stderr).strip()
print("Scanner        :", SCANNER)
print("Scanner version:", SCANNER_VERSION)

# Arguments shared by every configuration. JSON output is what Section 8 parses.
COMMON_ARGS = ["--format", "json", "--policy", POLICY]
if LENIENT:
    COMMON_ARGS.append("--lenient")
if ENABLE_OSV:
    COMMON_ARGS.append("--use-osv")

# Which exported folder each scope uses (Section 6 creates them).
SCOPE_FOLDER = {"SKILL.md only": "only", "Full folder": "full"}

SCAN_CONFIGS = [
    {"configuration": "SKILL.md only | default",    "scope": "SKILL.md only", "analyzer_mode": "default",    "report_file": "skill_only_default.json",    "extra_args": []},
    {"configuration": "SKILL.md only | behavioral", "scope": "SKILL.md only", "analyzer_mode": "behavioral", "report_file": "skill_only_behavioral.json", "extra_args": ["--use-behavioral"]},
    {"configuration": "Full folder | default",      "scope": "Full folder",   "analyzer_mode": "default",    "report_file": "full_folder_default.json",   "extra_args": []},
    {"configuration": "Full folder | behavioral",   "scope": "Full folder",   "analyzer_mode": "behavioral", "report_file": "full_folder_behavioral.json","extra_args": ["--use-behavioral"]},
]

if ENABLE_LLM_CONFIGS:
    if not os.environ.get("SKILL_SCANNER_LLM_API_KEY"):
        print("WARNING: ENABLE_LLM_CONFIGS is True but SKILL_SCANNER_LLM_API_KEY is not set; LLM configurations are NOT added.")
    else:
        llm_args = ["--use-llm", "--llm-provider", LLM_PROVIDER]
        SCAN_CONFIGS += [
            {"configuration": "SKILL.md only | llm", "scope": "SKILL.md only", "analyzer_mode": "llm", "report_file": "skill_only_llm.json", "extra_args": llm_args},
            {"configuration": "Full folder | llm",   "scope": "Full folder",   "analyzer_mode": "llm", "report_file": "full_folder_llm.json",  "extra_args": llm_args},
        ]
        print("LLM configurations added. Skill contents WILL be sent to the configured provider and billed.")

CONFIG_ORDER = [cfg["configuration"] for cfg in SCAN_CONFIGS]
display(pd.DataFrame(SCAN_CONFIGS).assign(common_args=" ".join(COMMON_ARGS)))

## 5 — Select the skills to scan

This section produces `selected_df`, one row per skill, which the export and scan sections consume.

`lookup_skill(repo, path)` joins each requested row to its **representative** (`rep.file_sha = a.file_sha AND
rep.dedup_primary = 1`) so copies resolve to the row that actually stores the text and supporting files. It also
pulls the GitSkills fields that matter for interpreting results later:

* `frontmatter_valid` — whether GitSkills could parse the YAML header (Cisco falls back to defaults when it cannot;
  about 13 % of distinct skills in the sample are affected),
* `sibling_count`, `has_scripts`, `has_references` — how much supporting context exists,
* `composition_fetched`, `composition_truncated` — whether GitSkills captured the folder listing completely.

In `search` mode nothing is selected: it previews matches so you can pick one for `exact` mode. The search
restricts to `dedup_primary = 1` so each distinct skill appears once, and uses `instr(lower(content), lower(?))`
for a case-insensitive substring match. On the full release this scan over 1M+ rows is slow; sampling should
use indexed metadata columns instead (Section 14).

**Output:** the selection table (or the search preview).

In [ ]:
def lookup_skill(repo, path):
    """Describe one skill and resolve it to the representative row that stores its text and files."""
    rows = sql("""
        SELECT a.repo_full_name AS repo, a.path AS skill_path, a.filename, a.dedup_primary,
               rep.repo_full_name AS rep_repo, rep.path AS rep_path,
               rep.name, rep.frontmatter_valid, rep.sibling_count, rep.has_scripts, rep.has_references,
               rep.composition_fetched, rep.composition_truncated, rep.file_sha
        FROM artifacts a
        JOIN artifacts rep ON rep.file_sha = a.file_sha AND rep.dedup_primary = 1
        WHERE a.repo_full_name = ? AND a.path = ?
    """, (repo, path))
    if rows.empty:
        raise ValueError(f"Skill not found in the database: {repo} / {path}")
    return rows.iloc[0]


def search_skills(keyword, limit):
    """Distinct skills whose SKILL.md text contains `keyword` (case-insensitive). Preview columns only."""
    return sql("""
        SELECT repo_full_name AS repo, path AS skill_path, name, sibling_count, has_scripts, has_references,
               frontmatter_valid, substr(content, 1, 100) AS preview
        FROM artifacts
        WHERE dedup_primary = 1 AND instr(lower(content), lower(?)) > 0
        ORDER BY repo_full_name, path
        LIMIT ?
    """, (keyword, limit))


if MODE == "pilot":
    selections = [PILOTS[PILOT]]
elif MODE == "exact":
    if not EXACT_REPO.strip() or not EXACT_PATH.strip():
        raise ValueError("Set EXACT_REPO and EXACT_PATH in Section 1")
    selections = [(EXACT_REPO.strip(), EXACT_PATH.strip())]
elif MODE == "search":
    matches = search_skills(SEARCH_KEYWORD, PREVIEW_LIMIT)
    print(f"Preview of up to {PREVIEW_LIMIT} distinct skills containing {SEARCH_KEYWORD!r} (nothing will be scanned):")
    display(matches)
    selections = []
else:  # batch
    matches = search_skills(SEARCH_KEYWORD, MAX_SKILLS)
    selections = list(matches[["repo", "skill_path"]].itertuples(index=False, name=None))

selected_df = pd.DataFrame([lookup_skill(repo, path) for repo, path in selections])

print(f"Selected {len(selected_df)} skill(s).")
if not selected_df.empty:
    copies = selected_df[selected_df["dedup_primary"] != 1]
    if not copies.empty:
        print(f"{len(copies)} selection(s) are copies; their text is taken from the representative row (rep_repo / rep_path).")
    display(selected_df[["repo", "skill_path", "name", "frontmatter_valid", "sibling_count", "has_scripts",
                         "has_references", "composition_fetched", "composition_truncated", "rep_repo", "rep_path"]])
elif MODE == "search":
    print("SEARCH MODE: preview only. Set MODE = 'exact' with a repo/path above, or MODE = 'batch', to scan.")

## 6 — Export each skill as files on disk

Cisco scans folders, so each selected skill is written under `results/cisco_dataset_runs/<database>/<skill_id>/`:

```
<skill_id>/
  only/SKILL.md            <- scope "SKILL.md only"
  full/SKILL.md            <- scope "Full folder": the SAME bytes as only/SKILL.md ...
  full/scripts/agent.py    <- ... plus every supporting file whose text GitSkills stored
  manifest.json            <- what was written, SHA-256 of every file, what was missing
  reports/*.json           <- Cisco reports (Section 7)
```

Design decisions, each a small function below so it can be read and tested on its own:

* **`skill_id_for`** builds a folder name from the skill's folder name plus a 12-character hash of
  `repo + path`, so two skills with the same folder name never collide and the name is stable across runs.
* **`safe_target_path`** rejects supporting-file names that could escape the folder (`..`, absolute paths,
  backslashes) or overwrite the instruction file. A *nested* `sub/SKILL.md` is allowed and recorded: some
  skill folders contain sub-skills, and Cisco treats those files as ordinary Markdown.
* **`write_exact_bytes`** writes the UTF-8 bytes of the stored text and refuses to overwrite a file whose
  bytes differ, so re-running the notebook can never silently change what an earlier report described.
  Comparing bytes (not decoded text) matters because 334 skills in the sample use Windows line endings.
* **`export_skill`** writes both scopes, asserts the two `SKILL.md` files are byte-identical (the only intended
  difference between scopes is the presence of supporting files), and decides **folder completeness**:
  complete means GitSkills fetched the folder listing without truncation **and** stored text for every file.
  Incomplete folders are not "safe"; they are a different situation and Section 7 skips their full scan by
  default.

**Output:** `exports` (one manifest dict per skill) and `exports_df`, a table of what was written.

In [ ]:
RUN_ROOT = RESULTS_ROOT / DB_TAG
RUN_ROOT.mkdir(parents=True, exist_ok=True)


def skill_id_for(repo, skill_path):
    """Stable, filesystem-safe folder name: <skill folder name>-<12 hex chars of sha256(repo + path)>."""
    folder_name = Path(skill_path).parent.name
    stem = re.sub(r"[^a-zA-Z0-9_-]", "-", folder_name)[:35].strip("-") or "skill"
    digest = hashlib.sha256(f"{repo}\n{skill_path}".encode()).hexdigest()[:12]
    return f"{stem}-{digest}"


def safe_target_path(folder, entry_name):
    """Where a supporting file goes inside `folder`, or raise if the name is unsafe.

    GitSkills sibling names are relative to the skill folder, e.g. "scripts/agent.py".
    """
    rel = Path(entry_name)
    if not entry_name or rel.is_absolute() or ".." in rel.parts or "\\" in entry_name:
        raise ValueError(f"Unsafe supporting-file path: {entry_name!r}")
    if len(rel.parts) == 1 and rel.name.lower() == "skill.md":
        # A top-level file named SKILL.md (any letter case; macOS file systems ignore case) would overwrite the instruction file.
        raise ValueError(f"Supporting file would overwrite SKILL.md: {entry_name!r}")
    target = folder / rel
    if not target.resolve().is_relative_to(folder.resolve()):
        raise ValueError(f"Path escapes the export folder: {entry_name!r}")
    return target


def write_exact_bytes(path, data):
    """Write `data` (bytes) to `path`; if the file exists with different bytes, refuse instead of overwriting."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes() != data:
            raise ValueError(f"File differs from an earlier export; refusing to overwrite: {path}")
    else:
        path.write_bytes(data)


def export_skill(skill):
    """Write the two scope folders for one selected skill and return its manifest (also saved as manifest.json)."""
    skill_id = skill_id_for(skill.repo, skill.skill_path)
    skill_dir = RUN_ROOT / skill_id
    folder_only, folder_full = skill_dir / "only", skill_dir / "full"

    text = sql("SELECT content FROM artifacts WHERE repo_full_name = ? AND path = ?", (skill.rep_repo, skill.rep_path))
    content = text.iloc[0]["content"] if not text.empty else None
    if not isinstance(content, str) or not content.strip():
        raise ValueError("SKILL.md text is not stored for this skill (or is empty)")
    skill_bytes = content.encode("utf-8")
    write_exact_bytes(folder_only / "SKILL.md", skill_bytes)
    write_exact_bytes(folder_full / "SKILL.md", skill_bytes)

    siblings = sql("""
        SELECT entry_name, entry_type, content, content_fetched
        FROM artifact_siblings
        WHERE repo_full_name = ? AND artifact_path = ?
        ORDER BY entry_name
    """, (skill.rep_repo, skill.rep_path))

    written, missing, nested_skill_files = [], [], []
    for item in siblings.itertuples(index=False):
        if item.entry_type != "file":
            continue                                   # directories are created implicitly
        target = safe_target_path(folder_full, item.entry_name)   # validate every name, even ones we cannot write
        if item.content_fetched != 1 or not isinstance(item.content, str):
            missing.append(item.entry_name)            # binary / oversized / capped folder: text not in GitSkills
            continue
        if Path(item.entry_name).name.lower() == "skill.md":
            nested_skill_files.append(item.entry_name)
        write_exact_bytes(target, item.content.encode("utf-8"))
        written.append(item.entry_name)

    if (folder_only / "SKILL.md").read_bytes() != (folder_full / "SKILL.md").read_bytes():
        raise AssertionError("SKILL.md bytes differ between scopes")   # should be impossible; checked anyway

    listing_complete = bool(pd.notna(skill.composition_fetched) and int(skill.composition_fetched) == 1
                            and pd.notna(skill.composition_truncated) and int(skill.composition_truncated) == 0)
    manifest = {
        "skill_id": skill_id,
        "repo": skill.repo, "skill_path": skill.skill_path, "original_filename": skill.filename,
        "representative_repo": skill.rep_repo, "representative_path": skill.rep_path, "file_sha": skill.file_sha,
        "database": str(DB_PATH), "database_sha256": DB_SHA256,
        "frontmatter_valid": int(skill.frontmatter_valid) if pd.notna(skill.frontmatter_valid) else None,
        "has_scripts": int(skill.has_scripts) if pd.notna(skill.has_scripts) else None,
        "has_references": int(skill.has_references) if pd.notna(skill.has_references) else None,
        "sibling_count_in_db": int(skill.sibling_count) if pd.notna(skill.sibling_count) else None,
        "listing_complete": listing_complete,
        "supporting_files_written": written,
        "missing_files": missing,
        "nested_skill_files": nested_skill_files,
        "folder_complete": listing_complete and not missing,
        "skill_md_sha256": hashlib.sha256(skill_bytes).hexdigest(),
        "files": [{"path": p.relative_to(folder_full).as_posix(), "bytes": p.stat().st_size, "sha256": sha256_of_file(p)}
                  for p in sorted(folder_full.rglob("*")) if p.is_file()],
        "exported_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    (skill_dir / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    return manifest

In [ ]:
exports, export_errors = [], []

for number, skill in enumerate(selected_df.itertuples(index=False), start=1):
    print(f"[{number}/{len(selected_df)}] {skill.repo} / {skill.skill_path}")
    try:
        manifest = export_skill(skill)
    except Exception as exc:          # one bad skill must not stop the batch; it is recorded, not hidden
        print(f"    EXPORT ERROR: {type(exc).__name__}: {str(exc)[:200]}")
        export_errors.append({"skill_id": skill_id_for(skill.repo, skill.skill_path), "repo": skill.repo,
                              "skill_path": skill.skill_path, "error": f"{type(exc).__name__}: {exc}"})
        continue
    exports.append(manifest)
    print(f"    wrote SKILL.md + {len(manifest['supporting_files_written'])} supporting file(s); "
          f"folder complete: {manifest['folder_complete']}"
          + (f"; missing text for {manifest['missing_files']}" if manifest["missing_files"] else "")
          + (f"; nested skill files: {manifest['nested_skill_files']}" if manifest["nested_skill_files"] else ""))

exports_df = pd.DataFrame([{
    "skill_id": m["skill_id"], "repo": m["repo"], "skill_path": m["skill_path"],
    "frontmatter_valid": m["frontmatter_valid"], "has_scripts": m["has_scripts"], "has_references": m["has_references"],
    "files_written": len(m["supporting_files_written"]), "files_missing": len(m["missing_files"]),
    "listing_complete": m["listing_complete"], "folder_complete": m["folder_complete"],
} for m in exports], columns=["skill_id", "repo", "skill_path", "frontmatter_valid", "has_scripts", "has_references",
                             "files_written", "files_missing", "listing_complete", "folder_complete"])

print(f"\nExported {len(exports)} skill(s) to {RUN_ROOT}" + (f"; {len(export_errors)} export error(s)" if export_errors else ""))
display(exports_df)

## 7 — Run the Cisco scanner

For every exported skill and every configuration the notebook runs one command, for example

```
skill-scanner scan results/.../full --format json --policy balanced --use-behavioral
```

`subprocess.run([...], capture_output=True, text=True, timeout=...)` starts that command as a separate
process, waits for it, and gives us its exit code, standard output (the JSON report) and standard error
(warnings). Running the command-line tool rather than importing Cisco's Python API keeps the scanner exactly as
other users run it and isolates us from its crashes.

Every run ends in exactly one **status**, and the statuses are *not* interchangeable:

| status | meaning | counts as "0 findings"? |
|---|---|---|
| `VALID` | report parsed and has the expected shape | only if `findings_count` is 0 |
| `SKIPPED_INCOMPLETE_FOLDER` | GitSkills did not store every supporting file; full scan not run | no |
| `TIMEOUT` | the scanner did not finish within `SCAN_TIMEOUT_SECONDS` | no |
| `SCANNER_ERROR` | the scanner exited without a JSON report | no |
| `INVALID_FORMAT` / `INVALID_JSON` | output exists but is not a report we recognise | no |
| `MISSING_REPORT` | `RUN_SCANS = False` and no saved report exists | no |
| `EXPORT_ERROR` | the skill could not be exported, so nothing was scanned | no |

Valid reports are saved **atomically** (written to a temporary file, then renamed) under `reports/`, so an
interrupted run never leaves a half-written JSON file. The complete original report is kept; Section 8 only
*reads* from it.

Besides the verdict and counts, each run row records what the report says about itself: `analyzers_used`,
`analyzers_failed` (Cisco 2.3.1 reports an internal rule-contract error for one behavioral rule; the finding
is still emitted, and we keep the note), the policy name and fingerprint, and the scanner's own
`scan_duration_seconds` next to our wall-clock time (which includes process start-up).

**Output:** `scan_runs_df` (one row per skill × configuration) and `reports` (the parsed JSON, in memory).

In [ ]:
def validate_report(obj):
    """Return None if `obj` looks like a Cisco single-skill report, otherwise a short reason."""
    if not isinstance(obj, dict):
        return "top level is not a JSON object"
    for key, expected_type in (("findings", list), ("is_safe", bool), ("max_severity", str)):
        if not isinstance(obj.get(key), expected_type):
            return f"missing or wrong type for '{key}'"
    return None


def run_scanner(folder, extra_args):
    """Run one scanner invocation on `folder`. Returns a dict with status, detail, exit code, timing and the report."""
    command = [SCANNER, "scan", str(folder), *COMMON_ARGS, *extra_args]
    started = time.perf_counter()
    try:
        proc = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                              timeout=SCAN_TIMEOUT_SECONDS, check=False)
    except subprocess.TimeoutExpired:
        return {"status": "TIMEOUT", "detail": f"no result within {SCAN_TIMEOUT_SECONDS} s", "exit_code": None,
                "wall_seconds": time.perf_counter() - started, "report": None, "command": command}
    wall = time.perf_counter() - started
    stderr_tail = proc.stderr.strip()[-400:]
    try:
        obj = json.loads(proc.stdout)
    except json.JSONDecodeError:
        return {"status": "SCANNER_ERROR", "detail": f"exit code {proc.returncode}; stderr: {stderr_tail or '(empty)'}",
                "exit_code": proc.returncode, "wall_seconds": wall, "report": None, "command": command}
    problem = validate_report(obj)
    if problem:
        return {"status": "INVALID_FORMAT", "detail": problem, "exit_code": proc.returncode,
                "wall_seconds": wall, "report": None, "command": command}
    detail = f"exit code {proc.returncode}" + (f"; stderr: {stderr_tail}" if stderr_tail else "")
    return {"status": "VALID", "detail": detail, "exit_code": proc.returncode, "wall_seconds": wall,
            "report": obj, "command": command}


def save_report(report, path):
    """Write the JSON report to a temporary file in the same folder, then rename: never a half-written report."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, prefix=".tmp_", suffix=".json", delete=False) as tmp:
        json.dump(report, tmp, indent=2)
        tmp.write("\n")
    Path(tmp.name).replace(path)


def load_report(path):
    """Read a saved report. Returns (report or None, status, detail)."""
    if not path.is_file():
        return None, "MISSING_REPORT", "no saved report for this configuration; set RUN_SCANS = True to create it"
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        return None, "INVALID_JSON", str(exc)
    problem = validate_report(obj)
    if problem:
        return None, "INVALID_FORMAT", problem
    return obj, "VALID", "loaded from disk"

In [ ]:
SCAN_RUN_COLUMNS = ["skill_id", "repo", "skill_path", "configuration", "scope", "analyzer_mode", "status", "status_detail",
                    "exit_code", "wall_seconds", "scan_seconds", "is_safe", "max_severity", "findings_count",
                    "folder_complete", "files_missing", "frontmatter_valid", "analyzers_used", "analyzers_failed",
                    "policy_name", "policy_fingerprint", "scanner_version", "report_file"]

scan_runs = []
reports = {}      # (skill_id, configuration) -> (report dict, folder that was scanned)

for number, manifest in enumerate(exports, start=1):
    skill_dir = RUN_ROOT / manifest["skill_id"]
    print(f"\n[{number}/{len(exports)}] {manifest['repo']} / {manifest['skill_path']}")
    for cfg in SCAN_CONFIGS:
        folder = skill_dir / SCOPE_FOLDER[cfg["scope"]]
        report_path = skill_dir / "reports" / cfg["report_file"]
        row = dict.fromkeys(SCAN_RUN_COLUMNS)
        row.update(skill_id=manifest["skill_id"], repo=manifest["repo"], skill_path=manifest["skill_path"],
                   configuration=cfg["configuration"], scope=cfg["scope"], analyzer_mode=cfg["analyzer_mode"],
                   folder_complete=manifest["folder_complete"], files_missing=len(manifest["missing_files"]),
                   frontmatter_valid=manifest["frontmatter_valid"], scanner_version=SCANNER_VERSION,
                   report_file=report_path.relative_to(ROOT).as_posix())
        report = None
        if RUN_SCANS:
            if cfg["scope"] == "Full folder" and not manifest["folder_complete"] and not ALLOW_INCOMPLETE_FOLDERS:
                row.update(status="SKIPPED_INCOMPLETE_FOLDER",
                           status_detail=f"text missing for {len(manifest['missing_files'])} file(s): {manifest['missing_files'][:5]}")
            else:
                result = run_scanner(folder, cfg["extra_args"])
                row.update(status=result["status"], status_detail=result["detail"], exit_code=result["exit_code"],
                           wall_seconds=round(result["wall_seconds"], 2))
                report = result["report"]
                if report is not None:
                    save_report(report, report_path)
        else:
            report, row["status"], row["status_detail"] = load_report(report_path)

        if report is not None:
            meta = report.get("scan_metadata") or {}
            row.update(is_safe=report["is_safe"], max_severity=report["max_severity"], findings_count=len(report["findings"]),
                       scan_seconds=report.get("scan_duration_seconds"),
                       analyzers_used="|".join(report.get("analyzers_used") or []),
                       analyzers_failed=json.dumps(report["analyzers_failed"]) if report.get("analyzers_failed") else None,
                       policy_name=meta.get("policy_name"), policy_fingerprint=meta.get("policy_fingerprint_sha256"))
            reports[(manifest["skill_id"], cfg["configuration"])] = (report, folder)
        scan_runs.append(row)
        shown = f"{row['findings_count']} findings, max {row['max_severity']}, is_safe={row['is_safe']}" if report else row["status_detail"]
        print(f"    {cfg['configuration']:<28} {row['status']:<26} {shown}")

for err in export_errors:
    row = dict.fromkeys(SCAN_RUN_COLUMNS)
    row.update(skill_id=err["skill_id"], repo=err["repo"], skill_path=err["skill_path"], configuration="(export)",
               status="EXPORT_ERROR", status_detail=err["error"], scanner_version=SCANNER_VERSION)
    scan_runs.append(row)

scan_runs_df = pd.DataFrame(scan_runs, columns=SCAN_RUN_COLUMNS)
if scan_runs_df.empty:
    print("No scans were run (no skills selected).")
else:
    print("\nRun status counts:")
    display(scan_runs_df["status"].value_counts().rename_axis("status").to_frame("runs"))
    fingerprints = scan_runs_df["policy_fingerprint"].dropna().unique()
    if len(fingerprints) > 1:
        print("WARNING: reports carry different policy fingerprints; results are not directly comparable:", fingerprints)

## 8 — Turn the reports into one findings table

A Cisco report is a nested JSON object: scan-level fields plus a list of findings, each with its own
`metadata` dictionary. Analysis needs a flat table with **one row per finding**, so this section walks each
report and copies out:

* the core fields Cisco gives every finding: `rule_id`, `category` (kept as `cisco_category`), `severity`,
  `analyzer`, `title`, `description`, `file_path`, `line_number`, `snippet`, `remediation`, and Cisco's own
  `finding_id`;
* the useful parts of `metadata`: the matched text and regex for signature rules, the files involved and
  `evidence` for cross-file behavioral rules, whether the match came from a code block in Markdown
  (`from_code_block`, `block_language`), and Cisco's taxonomy codes (`aitech`, `aisubtech`), plus the complete
  metadata as a JSON string so nothing is lost.

Three columns are **ours**, and the notebook says so wherever they appear:

* **`file_path` normalization.** Some behavioral findings carry an absolute path of the scanned folder and
  cross-file findings carry no path. We make paths relative to the scanned folder so a finding on
  `scripts/agent.py` has the same key in every scope and on every machine. `location` summarises it as
  `SKILL.md`, `supporting file`, or `whole skill (no file)`.
* **`finding_kind`** separates `security` findings from `metadata_quality` ones (missing license, vague
  description, name/length limits) and `scan_coverage` notes (binary files, low analyzability, load fallbacks,
  analysis-incomplete warnings). The lists are explicit below; everything not listed is treated as security.
  Cisco itself gives most of these `INFO` or `LOW` severity but mixes them into the same findings list.
* **`research_category`** maps Cisco's category (with a few rule-level overrides) onto the eight behavior
  categories of RQ1-B. Cisco has **no** rules for two of them (file-system modification as such, and account
  modification / sensitive external actions), which is itself an RQ1-C result: those categories are
  `not covered by Cisco` in the table rather than silently merged.

Finally, two keys make findings comparable across configurations (Section 11–12):
`location_key = rule_id | file_path | line_number` and `finding_key = location_key | hash(snippet or matched text)`.
Two findings are called *the same* only when the full `finding_key` matches, never because their severity or
category happens to match.

**Output:** `findings_df`. The display uses `redact()` to mask credential-like text in snippets; the CSV and the
raw JSON keep Cisco's evidence unmodified.

In [ ]:
# --- Our classification of Cisco rules (edit here; these are research decisions, not Cisco's) -------------
METADATA_QUALITY_RULES = {
    "MANIFEST_MISSING_LICENSE", "MANIFEST_INVALID_NAME", "MANIFEST_DESCRIPTION_TOO_LONG", "MANIFEST_METADATA_INCOMPLETE",
    "MISSING_STRUCTURED_METADATA", "INVALID_STRUCTURED_METADATA", "MALFORMED_YAML_FRONTMATTER",
    "SKILL_METADATA_SIZE_LIMIT_EXCEEDED", "SOCIAL_ENG_VAGUE_DESCRIPTION",
    "TRIGGER_VAGUE_DESCRIPTION", "TRIGGER_OVERLY_GENERIC", "TRIGGER_DESCRIPTION_TOO_SHORT",
}
SCAN_COVERAGE_RULES = {
    "LOW_ANALYZABILITY", "UNANALYZABLE_BINARY", "BINARY_FILE_DETECTED", "ARCHIVE_FILE_DETECTED", "PYCACHE_FILES_DETECTED",
    "OVERSIZED_FILE", "EXCESSIVE_FILE_COUNT", "SKILL_LOAD_FALLBACK_USED", "SKILL_LOAD_FAILED", "SKILL_LOAD_REJECTED_LIMIT",
    "SYMLINK_NOT_SCANNED", "UNICODE_ANALYSIS_INCOMPLETE", "BEHAVIOR_DATAFLOW_ANALYSIS_INCOMPLETE",
    "BYTECODE_ANALYSIS_UNAVAILABLE", "ARCHIVE_EXTRACTION_FAILED",
    "LLM_ANALYSIS_FAILED", "LLM_CONTEXT_BUDGET_EXCEEDED", "META_CONTEXT_BUDGET_EXCEEDED",
}

RESEARCH_CATEGORIES = [
    "Credential / secret access or exposure",
    "External network communication / data exfiltration",
    "Command execution / dangerous shell operations",
    "File-system access or modification",
    "Prompt injection / instruction manipulation",
    "Excessive permissions / privilege escalation",
    "Supply-chain / dependency risks",
    "Account modification / sensitive external actions",
]
# Cisco category -> research category. Categories not listed become "Other (<cisco category>)".
RESEARCH_CATEGORY_BY_CISCO_CATEGORY = {
    "hardcoded_secrets": RESEARCH_CATEGORIES[0],
    "data_exfiltration": RESEARCH_CATEGORIES[1],
    "command_injection": RESEARCH_CATEGORIES[2],
    "prompt_injection": RESEARCH_CATEGORIES[4],
    "unicode_steganography": RESEARCH_CATEGORIES[4],
    "social_engineering": RESEARCH_CATEGORIES[4],
    "unauthorized_tool_use": RESEARCH_CATEGORIES[5],
    "autonomy_abuse": RESEARCH_CATEGORIES[5],
    "tool_chaining_abuse": RESEARCH_CATEGORIES[5],
    "transitive_trust_abuse": RESEARCH_CATEGORIES[5],
    "skill_discovery_abuse": RESEARCH_CATEGORIES[5],
    "supply_chain_attack": RESEARCH_CATEGORIES[6],
}
# Rule-ID prefixes that override the category mapping (checked first, longest match wins).
RESEARCH_CATEGORY_BY_RULE_PREFIX = {
    "SECRET_": RESEARCH_CATEGORIES[0],                        # hardcoded secrets
    "DATA_EXFIL_SENSITIVE_FILE": RESEARCH_CATEGORIES[0],      # reads ~/.ssh, .env, credential files (Cisco files it under exfiltration)
    "DATA_EXFIL_ENV": RESEARCH_CATEGORIES[0],                 # environment-variable dumps
    "BEHAVIOR_CREDENTIAL_FILE_ACCESS": RESEARCH_CATEGORIES[0],
    "BEHAVIOR_ENV_VAR_HARVESTING": RESEARCH_CATEGORIES[0],
    "PATH_TRAVERSAL": RESEARCH_CATEGORIES[3],                 # path traversal is the closest Cisco gets to file-system abuse
    "GLOB_HIDDEN_FILE_TARGETING": RESEARCH_CATEGORIES[3],
    "PIPELINE_TAINT_FLOW": RESEARCH_CATEGORIES[2],
    "MDBLOCK_PYTHON_SUBPROCESS": RESEARCH_CATEGORIES[2],
    "MDBLOCK_PYTHON_EVAL_EXEC": RESEARCH_CATEGORIES[2],
    "BEHAVIOR_EVAL_SUBPROCESS": RESEARCH_CATEGORIES[2],
    "BEHAVIOR_BASH_TAINT_FLOW": RESEARCH_CATEGORIES[2],
    "SUPPLY_CHAIN_": RESEARCH_CATEGORIES[6],
    "TOOL_ABUSE_SYSTEM_PACKAGE_INSTALL": RESEARCH_CATEGORIES[6],
    "TOOL_ABUSE_UNTRUSTED_PACKAGE_SOURCE": RESEARCH_CATEGORIES[6],
}
RESEARCH_CATEGORIES_NOT_COVERED_BY_CISCO = [RESEARCH_CATEGORIES[7]]   # no Cisco rule family maps here; documented gap


def classify_finding_kind(rule_id):
    if rule_id in METADATA_QUALITY_RULES:
        return "metadata_quality"
    if rule_id in SCAN_COVERAGE_RULES:
        return "scan_coverage"
    return "security"


def research_category(rule_id, cisco_category):
    for prefix in sorted(RESEARCH_CATEGORY_BY_RULE_PREFIX, key=len, reverse=True):
        if str(rule_id).startswith(prefix):
            return RESEARCH_CATEGORY_BY_RULE_PREFIX[prefix]
    return RESEARCH_CATEGORY_BY_CISCO_CATEGORY.get(cisco_category, f"Other ({cisco_category})")


def normalize_file_path(file_path, scanned_folder):
    """Make Cisco's file_path relative to the scanned folder; None stays None; unknown absolute paths are kept as given."""
    if not file_path:
        return None
    path = Path(file_path)
    if path.is_absolute():
        try:
            return path.resolve().relative_to(scanned_folder.resolve()).as_posix()
        except ValueError:
            return file_path
    return path.as_posix()


def describe_location(file_path):
    if file_path is None:
        return "whole skill (no file)"
    return "SKILL.md" if file_path == "SKILL.md" else "supporting file"


SECRET_PATTERNS = [
    re.compile(r"(?i)\b(password|passwd|secret|token|api[_-]?key|authorization)(\s*[=:]\s*)\S+"),
    re.compile(r"://[^/\s:@]+:[^@\s]+@"),                       # user:password@host in URLs
    re.compile(r"\b(AKIA|ASIA|sk-|ghp_|gho_|xox[bap]-|AIza)[A-Za-z0-9_\-]{8,}"),
]


def redact(text, max_len=100):
    """Mask credential-looking text for DISPLAY only (the CSV and JSON keep the original evidence)."""
    if not isinstance(text, str):
        return text
    for pattern in SECRET_PATTERNS:
        text = pattern.sub(lambda m: (m.group(1) + m.group(2) if m.lastindex and m.lastindex >= 2 else "") + "[REDACTED]", text)
    text = text.replace("\n", " ")
    return text if len(text) <= max_len else text[:max_len] + "..."

In [ ]:
METADATA_FIELDS_TO_KEEP = ["matched_text", "matched_pattern", "signature_context", "scanner_category", "aitech", "aitech_name",
                           "aisubtech", "aisubtech_name", "files_involved", "evidence", "from_code_block", "block_language",
                           "deduped_rule_ids", "threat_type"]
FINDING_COLUMNS = (["skill_id", "repo", "skill_path", "configuration", "scope", "analyzer_mode", "finding_id", "rule_id",
                    "cisco_category", "severity", "analyzer", "title", "description", "file_path", "line_number", "location",
                    "snippet", "remediation", "finding_kind", "research_category", "location_key", "finding_key"]
                   + [f"meta_{name}" for name in METADATA_FIELDS_TO_KEEP] + ["metadata_json"])


def as_cell_value(value):
    """Lists/dicts become JSON text so they survive a CSV round trip; scalars pass through."""
    return json.dumps(value, sort_keys=True) if isinstance(value, (list, dict)) else value


def flatten_findings(report, run, scanned_folder):
    """One flat row per finding in `report`; `run` is the matching scan_runs row (identity + configuration)."""
    rows = []
    for finding in report["findings"]:
        if not isinstance(finding, dict):
            continue
        meta = finding.get("metadata") or {}
        file_path = normalize_file_path(finding.get("file_path"), scanned_folder)
        evidence = finding.get("snippet") or meta.get("matched_text") or finding.get("title") or ""
        location_key = f"{finding.get('rule_id')}|{file_path}|{finding.get('line_number')}"
        row = {
            "skill_id": run["skill_id"], "repo": run["repo"], "skill_path": run["skill_path"],
            "configuration": run["configuration"], "scope": run["scope"], "analyzer_mode": run["analyzer_mode"],
            "finding_id": finding.get("id"), "rule_id": finding.get("rule_id"), "cisco_category": finding.get("category"),
            "severity": finding.get("severity"), "analyzer": finding.get("analyzer"),
            "title": finding.get("title"), "description": finding.get("description"),
            "file_path": file_path, "line_number": finding.get("line_number"), "location": describe_location(file_path),
            "snippet": finding.get("snippet"), "remediation": finding.get("remediation"),
            "finding_kind": classify_finding_kind(finding.get("rule_id")),
            "research_category": research_category(finding.get("rule_id"), finding.get("category")),
            "location_key": location_key,
            "finding_key": f"{location_key}|{hashlib.sha256(str(evidence).encode()).hexdigest()[:10]}",
            "metadata_json": json.dumps(meta, sort_keys=True, default=str),
        }
        for name in METADATA_FIELDS_TO_KEEP:
            row[f"meta_{name}"] = as_cell_value(meta.get(name))
        rows.append(row)
    return rows


finding_rows = []
for run in scan_runs:
    if (run["skill_id"], run["configuration"]) in reports:
        report, scanned_folder = reports[(run["skill_id"], run["configuration"])]
        finding_rows.extend(flatten_findings(report, run, scanned_folder))
findings_df = pd.DataFrame(finding_rows, columns=FINDING_COLUMNS)

SEVERITY_ORDER = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "INFO"]
findings_df["severity"] = pd.Categorical(findings_df["severity"], categories=SEVERITY_ORDER, ordered=True)
findings_df["configuration"] = pd.Categorical(findings_df["configuration"], categories=CONFIG_ORDER, ordered=True)

print(f"{len(findings_df)} finding rows from {len(reports)} valid report(s).")
if not findings_df.empty:
    print("By kind:", findings_df["finding_kind"].value_counts().to_dict())
    preview = findings_df[["skill_id", "configuration", "severity", "rule_id", "cisco_category", "analyzer", "file_path",
                           "line_number", "finding_kind", "snippet"]].copy()
    preview["snippet"] = preview["snippet"].map(redact)
    display(preview.sort_values(["skill_id", "configuration", "severity"]).reset_index(drop=True))

## 9 — Save the structured results

Three files per run folder, next to the per-skill folders:

| file | contents | used by |
|---|---|---|
| `scan_summary.csv` | one row per skill × configuration (`scan_runs_df`) | reliability analysis, later aggregation |
| `individual_findings.csv` | one row per finding (`findings_df`) | all finding-level analysis; later comparison with other scanners |
| `run_metadata.json` | database path and hash, scanner version, policy, arguments, settings, selection, status counts | the reproducibility record |

The original Cisco reports stay under each `<skill_id>/reports/`; the CSVs never replace them. Re-running the
notebook on the same database overwrites these three run-level files with the current run's content; the
per-skill reports are only rewritten when `RUN_SCANS = True`.

In [ ]:
SUMMARY_CSV = RUN_ROOT / "scan_summary.csv"
FINDINGS_CSV = RUN_ROOT / "individual_findings.csv"
METADATA_JSON = RUN_ROOT / "run_metadata.json"

scan_runs_df.to_csv(SUMMARY_CSV, index=False)
findings_df.to_csv(FINDINGS_CSV, index=False)

run_metadata = {
    "run_finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "database": {"path": str(DB_PATH), "tag": DB_TAG, "sha256": DB_SHA256},
    "scanner": {"executable": SCANNER, "version": SCANNER_VERSION, "policy": POLICY, "common_args": COMMON_ARGS,
                "configurations": [{k: v for k, v in cfg.items()} for cfg in SCAN_CONFIGS],
                "policy_fingerprints_seen": sorted(scan_runs_df["policy_fingerprint"].dropna().unique().tolist())},
    "settings": {"MODE": MODE, "PILOT": PILOT, "EXACT_REPO": EXACT_REPO, "EXACT_PATH": EXACT_PATH, "SEARCH_KEYWORD": SEARCH_KEYWORD,
                 "MAX_SKILLS": MAX_SKILLS, "RUN_SCANS": RUN_SCANS, "LENIENT": LENIENT, "ALLOW_INCOMPLETE_FOLDERS": ALLOW_INCOMPLETE_FOLDERS,
                 "SCAN_TIMEOUT_SECONDS": SCAN_TIMEOUT_SECONDS, "ENABLE_OSV": ENABLE_OSV, "ENABLE_LLM_CONFIGS": ENABLE_LLM_CONFIGS},
    "selection": [{"repo": r, "skill_path": p} for r, p in selections],
    "skills_exported": [m["skill_id"] for m in exports],
    "export_errors": export_errors,
    "run_status_counts": scan_runs_df["status"].value_counts().to_dict(),
    "finding_rows": int(len(findings_df)),
}
METADATA_JSON.write_text(json.dumps(run_metadata, indent=2, default=str) + "\n", encoding="utf-8")

for path in (SUMMARY_CSV, FINDINGS_CSV, METADATA_JSON):
    print(f"saved {path.relative_to(ROOT)}  ({path.stat().st_size:,} bytes)")

## 10 — Analysis A: what security findings does Cisco report?

The tables below use only `VALID` runs, and most of them only `security` findings. Two counts are always
kept apart: the **number of findings** (rows) and the **number of distinct skills** with at least one such
finding, because one script can produce ten findings for the same rule.

* **A1** — the per-skill verdict matrix: findings, maximum severity and `is_safe` per configuration.
* **A2** — findings by kind per configuration: how much of the output is security vs metadata quality vs coverage notes.
* **A3** — security findings by Cisco category and by our research category (findings and distinct skills).
* **A4** — the rules that fire most often.
* **A5** — severity distribution per configuration.
* **A6** — where findings point: `SKILL.md`, a supporting file, or the whole skill.
* **A7** — the metadata-quality and coverage findings, listed so nobody mistakes them for security results.

Chart colors follow a fixed order per configuration so the same configuration has the same color in every figure.

In [ ]:
# Shared helpers for Sections 10-13 --------------------------------------------------------------------------
valid_runs = scan_runs_df[scan_runs_df["status"] == "VALID"].copy()
security = findings_df[findings_df["finding_kind"] == "security"].copy()

SERIES_COLORS = dict(zip(CONFIG_ORDER, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
SEVERITY_COLORS = {"INFO": "#86b6ef", "LOW": "#5598e7", "MEDIUM": "#2a78d6", "HIGH": "#1c5cab", "CRITICAL": "#0d366b"}


def count_table(df, rows, columns=None, unit="findings"):
    """Pivot `df` into counts. unit="findings" counts rows; unit="skills" counts distinct skill_id."""
    if df.empty:
        return pd.DataFrame()
    if unit == "skills":
        table = df.pivot_table(index=rows, columns=columns, values="skill_id", aggfunc="nunique", fill_value=0, observed=True)
    else:
        table = df.pivot_table(index=rows, columns=columns, values="finding_key", aggfunc="count", fill_value=0, observed=True)
    if columns == "configuration":
        # Keep every configuration that has a valid run, even with zero findings, so "0" stays visible.
        present = set(valid_runs["configuration"])
        table = table.reindex(columns=[c for c in CONFIG_ORDER if c in present], fill_value=0)
    return table.astype(int)


def grouped_bar_chart(table, title, xlabel, colors=None):
    """Horizontal grouped bars: one group per table row, one bar per column. Counts are written at the bar ends."""
    if table.empty or int(table.to_numpy().sum()) == 0:
        print(f"(no data to plot for: {title})")
        return
    rows, series = list(table.index), list(table.columns)
    bar_h = 0.8 / len(series)
    fig, ax = plt.subplots(figsize=(9, 0.9 + 0.45 * len(rows) * len(series)))
    for i, name in enumerate(series):
        y = [r - 0.4 + bar_h * (i + 0.5) for r in range(len(rows))]
        values = table[name].to_numpy()
        color = (colors or SERIES_COLORS).get(name, "#777777")
        ax.barh(y, values, height=bar_h * 0.9, color=color, label=str(name))
        for yy, v in zip(y, values):
            if v:
                ax.text(v + table.to_numpy().max() * 0.01, yy, str(int(v)), va="center", fontsize=8, color="#444444")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([str(r)[:60] for r in rows], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_xlim(0, table.to_numpy().max() * 1.12)          # room for the count labels
    ax.set_title(title, loc="left", fontsize=11)
    ax.legend(fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0))   # outside the plot, never over bars
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis="x", color="#dddddd", linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()


if valid_runs.empty:
    print("No VALID runs: the analysis sections will print notices instead of tables. Check Section 7 statuses.")
else:
    print(f"{len(valid_runs)} valid run(s) covering {valid_runs['skill_id'].nunique()} skill(s); "
          f"{len(security)} security finding row(s), {len(findings_df) - len(security)} metadata/coverage row(s).")

In [ ]:
# A1 - verdict matrix per skill and configuration
if not valid_runs.empty:
    a1 = valid_runs.pivot_table(index=["repo", "skill_path"], columns="configuration",
                                values=["findings_count", "max_severity", "is_safe"], aggfunc="first", observed=True)
    a1 = a1.reindex(columns=CONFIG_ORDER, level=1)
    print("A1  Findings, maximum severity and is_safe per configuration (all finding kinds)")
    display(a1)

    # A2 - what kind of findings each configuration produced
    print("\nA2  Findings by kind (rows) and configuration (columns)")
    display(count_table(findings_df, "finding_kind", "configuration"))

In [ ]:
# A3 - security findings by category: number of findings AND number of distinct skills
if not security.empty:
    print("A3a  Security findings by Cisco category (number of findings)")
    display(count_table(security, "cisco_category", "configuration"))
    print("\nA3b  Security findings by Cisco category (number of distinct skills with >= 1 finding)")
    display(count_table(security, "cisco_category", "configuration", unit="skills"))
    print("\nA3c  Security findings by research category (our mapping, see Section 8)")
    a3c = count_table(security, "research_category", "configuration")
    # Show research categories with zero findings too, so coverage gaps are visible.
    a3c = a3c.reindex(RESEARCH_CATEGORIES + [c for c in a3c.index if c not in RESEARCH_CATEGORIES], fill_value=0)
    display(a3c)
    print("Not covered by any Cisco rule family:", RESEARCH_CATEGORIES_NOT_COVERED_BY_CISCO)
    grouped_bar_chart(count_table(security, "cisco_category", "configuration"),
                      "Security findings by Cisco category and configuration", "number of findings")
else:
    print("No security findings in the valid reports.")

In [ ]:
# A4 - rules that fire most often; A5 - severity; A6 - location
if not security.empty:
    a4 = (security.groupby(["rule_id", "cisco_category", "analyzer"], observed=True)
          .agg(findings=("finding_key", "size"), skills=("skill_id", "nunique"),
               configurations=("configuration", lambda s: ", ".join(sorted(set(map(str, s))))),
               severities=("severity", lambda s: ", ".join(sorted(set(map(str, s))))))
          .sort_values("findings", ascending=False).reset_index())
    print("A4  Cisco rules by number of security findings (and distinct skills)")
    display(a4)

    print("\nA5  Severity of security findings per configuration")
    a5 = count_table(security, "configuration", "severity")
    a5 = a5.reindex(columns=[s for s in SEVERITY_ORDER if s in a5.columns])
    display(a5)
    grouped_bar_chart(a5, "Severity of security findings per configuration", "number of findings", colors=SEVERITY_COLORS)

    print("\nA6  Where security findings point (SKILL.md, a supporting file, or the whole skill)")
    display(count_table(security, "location", "configuration"))
    supporting = security[security["location"] == "supporting file"]
    if not supporting.empty:
        print("\nA6b  Supporting files with security findings")
        display(supporting.groupby(["skill_id", "file_path"], observed=True)
                .agg(findings=("finding_key", "size"), rules=("rule_id", lambda s: ", ".join(sorted(set(s)))))
                .sort_values("findings", ascending=False))

In [ ]:
# A7 - metadata-quality and scan-coverage findings, listed separately
non_security = findings_df[findings_df["finding_kind"] != "security"]
if non_security.empty:
    print("A7  No metadata-quality or scan-coverage findings in the valid reports.")
else:
    print("A7  Metadata-quality and scan-coverage findings (not security results)")
    display(non_security[["skill_id", "configuration", "finding_kind", "severity", "rule_id", "title", "file_path"]]
            .sort_values(["skill_id", "configuration"]).reset_index(drop=True))

## 11 — Analysis B: does the full skill folder change what Cisco finds? (RQ1-A)

For each skill and each analyzer mode we compare the `SKILL.md only` run with the `Full folder` run, **finding
by finding**, using the keys from Section 8:

| change | rule |
|---|---|
| `unchanged` | same `finding_key` (rule, file, line, evidence) in both scopes |
| `changed` | same rule at the same file and line, but different evidence or severity (the earlier version is shown as `changed (previous version)`) |
| `new` | only in the full-folder scan |
| `removed` | only in the SKILL.md-only scan |

A comparison is **valid** only when both runs are `VALID`; skills whose full folder was skipped as incomplete do
not appear here (Section 13 counts them). The comparison uses security findings only, so a missing-license note
never counts as "context changed the result".

Two things to look for when reading the tables:

* New findings located in supporting files show behavior that `SKILL.md` alone could not reveal (RQ1-A, first
  question). New findings located *in `SKILL.md`* with the full folder present (for example
  `TOOL_ABUSE_UNDECLARED_NETWORK` in the canary-tokens pilot) show Cisco **re-interpreting the instructions in
  the light of the scripts** — the second RQ1-A question, where context changes interpretation rather than
  adding an operation.
* Removed findings would mean the extra context made Cisco drop a warning. Cisco has no "explain away" logic
  in its deterministic analyzers; if removals appear, inspect them manually before claiming that.

In [ ]:
def compare_findings(before, after):
    """Finding-level diff of two tables for the same skill. Returns `after` rows with a `change` column,
    plus the `before` rows that disappeared ("removed") or were replaced ("changed (previous version)")."""
    before_keys, after_keys = set(before["finding_key"]), set(after["finding_key"])
    before_locs, after_locs = set(before["location_key"]), set(after["location_key"])
    rows = []
    for _, f in after.iterrows():
        change = "unchanged" if f.finding_key in before_keys else ("changed" if f.location_key in before_locs else "new")
        rows.append({**f.to_dict(), "change": change})
    for _, f in before.iterrows():
        if f.finding_key not in after_keys:
            rows.append({**f.to_dict(), "change": "changed (previous version)" if f.location_key in after_locs else "removed"})
    return pd.DataFrame(rows, columns=list(after.columns) + ["change"])


def paired_comparison(table, fixed_col, vary_col, before_value, after_value):
    """Run compare_findings for every (skill, fixed_col value) where both runs are VALID.
    Returns (diff table, list of valid pairs)."""
    diffs, pairs = [], []
    for (skill_id, fixed), runs in valid_runs.groupby(["skill_id", fixed_col]):
        available = set(runs[vary_col])
        if not {before_value, after_value} <= available:
            continue
        pairs.append({"skill_id": skill_id, fixed_col: fixed})
        subset = table[(table["skill_id"] == skill_id) & (table[fixed_col] == fixed)]
        diff = compare_findings(subset[subset[vary_col] == before_value], subset[subset[vary_col] == after_value])
        diff["skill_id"], diff[fixed_col] = skill_id, fixed
        diffs.append(diff)
    columns = list(table.columns) + ["change"]
    return (pd.concat(diffs, ignore_index=True) if diffs else pd.DataFrame(columns=columns)), pairs


context_diff, context_pairs = paired_comparison(security, fixed_col="analyzer_mode", vary_col="scope",
                                                before_value="SKILL.md only", after_value="Full folder")
print(f"Valid context comparisons (both scopes VALID): {len(context_pairs)} skill × analyzer-mode pair(s)")
if context_diff.empty:
    print("Nothing to compare (no valid pairs, or no security findings in them).")
else:
    print("\nB1  Change counts per skill and analyzer mode (SKILL.md only -> Full folder)")
    b1 = context_diff.pivot_table(index=["skill_id", "analyzer_mode"], columns="change", values="finding_key",
                                  aggfunc="count", fill_value=0, observed=True)
    display(b1)

    skills_with_new = context_diff[context_diff["change"] == "new"].groupby("analyzer_mode", observed=True)["skill_id"].nunique()
    print("\nB2  Distinct skills that gained security findings with the full folder, per analyzer mode:")
    display(skills_with_new.to_frame("skills_with_new_findings"))

In [ ]:
if not context_diff.empty:
    changed = context_diff[context_diff["change"] != "unchanged"].copy()
    changed["snippet"] = changed["snippet"].map(redact)
    print("B3  Every finding that is new, changed or removed when the full folder is scanned")
    display(changed[["skill_id", "analyzer_mode", "change", "severity", "rule_id", "cisco_category", "analyzer",
                     "location", "file_path", "line_number", "title", "snippet"]]
            .sort_values(["skill_id", "analyzer_mode", "change", "file_path"]).reset_index(drop=True))

    new_only = context_diff[context_diff["change"] == "new"]
    if not new_only.empty:
        print("\nB4  Which files contribute the new findings (location = SKILL.md means context changed how the instructions were read)")
        display(new_only.groupby(["analyzer_mode", "location", "file_path"], dropna=False, observed=True)
                .agg(findings=("finding_key", "size"), skills=("skill_id", "nunique"),
                     rules=("rule_id", lambda s: ", ".join(sorted(set(s))))).reset_index())

    per_skill_scope = count_table(security[security["skill_id"].isin({p["skill_id"] for p in context_pairs})],
                                  "skill_id", "configuration")
    grouped_bar_chart(per_skill_scope, "Security findings per skill: SKILL.md only vs full folder, by configuration",
                      "number of security findings")

## 12 — Analysis C: what does the behavioral analyzer add? (analyzer part of RQ1-C)

Same finding-level diff, now holding the scope fixed and comparing `default` with `--use-behavioral`.
The behavioral analyzer only ever **adds** analyzers (the default ones still run), so `removed` should be
empty; if it is not, Cisco's output de-duplication (`finding_output.dedupe_same_issue_per_location`, which
prefers the behavioral version of a finding at the same location) is the likely reason, and the pair shows up
as `changed`.

* **C1** — change counts per skill and scope.
* **C2** — the findings only behavioral analysis produced, with their `analyzer` label.
* **C3** — security findings by Cisco category and analyzer label, so you can see which categories each
  analyzer covers in practice.

In [ ]:
analyzer_diff, analyzer_pairs = paired_comparison(security, fixed_col="scope", vary_col="analyzer_mode",
                                                  before_value="default", after_value="behavioral")
print(f"Valid analyzer comparisons (both modes VALID): {len(analyzer_pairs)} skill × scope pair(s)")
if analyzer_diff.empty:
    print("Nothing to compare.")
else:
    print("\nC1  Change counts per skill and scope (default -> behavioral)")
    display(analyzer_diff.pivot_table(index=["skill_id", "scope"], columns="change", values="finding_key",
                                      aggfunc="count", fill_value=0, observed=True))

    added = analyzer_diff[analyzer_diff["change"].isin(["new", "changed"])].copy()
    added["snippet"] = added["snippet"].map(redact)
    print("\nC2  Findings that only appear (or change) with --use-behavioral")
    display(added[["skill_id", "scope", "change", "severity", "rule_id", "cisco_category", "analyzer", "file_path",
                   "line_number", "title", "snippet"]].sort_values(["skill_id", "scope", "file_path"]).reset_index(drop=True))

    print("\nC3  Security findings by Cisco category and the analyzer that emitted them (all valid runs)")
    display(count_table(security, "cisco_category", "analyzer"))
    print("\nC3b  ... and by configuration and analyzer")
    display(count_table(security, ["configuration", "analyzer"]))

## 13 — Analysis D: how reliable are these results?

Before any number above goes into a paper, this section answers: how many scans actually succeeded, how many
folders could not be rebuilt, which comparisons were valid, and how many findings still need a human.

* **D1** — run status counts per configuration.
* **D2** — folder completeness and frontmatter validity of the exported skills.
* **D3** — analyzer failures Cisco reported inside otherwise valid reports.
* **D4** — which comparisons are valid (and which skills lack data for them).
* **D5** — findings that require manual interpretation. Every security finding does; the table counts them per
  skill and severity, and the cell writes `manual_review_template.csv`: one row per distinct security finding
  (rule, file, line) with empty columns for a reviewer's interpretation (`observed operation` /
  `scanner warning only` / `potential vulnerability` / `confirmed malicious` / `false positive`) and notes.
  Manual verdicts stay in that file; they are never merged back into Cisco's output.

The comparison tables are also saved as `context_comparison.csv` and `analyzer_comparison.csv`.

In [ ]:
if scan_runs_df.empty:
    print("No runs to assess.")
else:
    print("D1  Run status per configuration")
    display(scan_runs_df.pivot_table(index="configuration", columns="status", values="skill_id", aggfunc="count",
                                     fill_value=0, observed=True).reindex([c for c in CONFIG_ORDER if c in set(scan_runs_df["configuration"])]))

    print("\nD2  Exported skills: folder completeness and frontmatter validity")
    display(exports_df[["skill_id", "files_written", "files_missing", "listing_complete", "folder_complete", "frontmatter_valid"]])
    print(f"    complete folders: {int(exports_df['folder_complete'].sum())} / {len(exports_df)}"
          f"    skills with invalid frontmatter: {int((exports_df['frontmatter_valid'] == 0).sum())}"
          f"    export errors: {len(export_errors)}")

    failed = scan_runs_df.dropna(subset=["analyzers_failed"])
    print(f"\nD3  Reports with analyzer failures noted by Cisco: {len(failed)}")
    if not failed.empty:
        display(failed[["skill_id", "configuration", "analyzers_failed"]])

    print("\nD4  Valid comparisons")
    all_skills = set(exports_df["skill_id"])
    context_ok = {p["skill_id"] for p in context_pairs}
    analyzer_ok = {p["skill_id"] for p in analyzer_pairs}
    display(pd.DataFrame({
        "comparison": ["SKILL.md only vs Full folder (RQ1-A)", "default vs behavioral (RQ1-C)"],
        "skill x pair count": [len(context_pairs), len(analyzer_pairs)],
        "skills with at least one valid pair": [len(context_ok), len(analyzer_ok)],
        "skills lacking data": [sorted(all_skills - context_ok), sorted(all_skills - analyzer_ok)],
    }))

In [ ]:
if not security.empty:
    print("D5  Security findings that need manual interpretation, per skill and severity (findings / distinct locations)")
    d5 = (security.groupby(["skill_id", "severity"], observed=True)
          .agg(findings=("finding_key", "size"), distinct_locations=("location_key", "nunique")).reset_index())
    display(d5.pivot_table(index="skill_id", columns="severity", values=["findings", "distinct_locations"], fill_value=0, observed=True))

    review = (security.sort_values("severity")
              .groupby(["skill_id", "repo", "skill_path", "rule_id", "file_path", "line_number"], dropna=False, observed=True)
              .agg(severity=("severity", "first"), cisco_category=("cisco_category", "first"),
                   research_category=("research_category", "first"), analyzer=("analyzer", "first"),
                   title=("title", "first"), snippet=("snippet", "first"),
                   seen_in_configurations=("configuration", lambda s: " ; ".join(sorted(set(map(str, s))))))
              .reset_index())
    review["reviewer"] = ""
    review["interpretation"] = ""      # observed operation | scanner warning only | potential vulnerability | confirmed malicious | false positive
    review["legitimate_purpose_in_context"] = ""   # e.g. "credential manager reads secrets by design"
    review["notes"] = ""
    REVIEW_CSV = RUN_ROOT / "manual_review_template.csv"
    review.to_csv(REVIEW_CSV, index=False)
    print(f"\n{len(review)} distinct security finding locations written to {REVIEW_CSV.relative_to(ROOT)} for manual review.")

context_diff.to_csv(RUN_ROOT / "context_comparison.csv", index=False)
analyzer_diff.to_csv(RUN_ROOT / "analyzer_comparison.csv", index=False)
print("saved context_comparison.csv and analyzer_comparison.csv in", RUN_ROOT.relative_to(ROOT))

## 14 — Interpretation, limitations, and what comes next

### Reading the results

* **A finding is an observed operation plus a rule match.** `DATA_EXFIL_NETWORK_REQUESTS` on
  `requests.post(...)` in a webhook-alerting skill is correct detection of a legitimate operation. Whether it is
  a vulnerability is a manual judgement recorded in `manual_review_template.csv`, never in Cisco's columns.
* **`is_safe` is coarse.** It only says "no CRITICAL/HIGH". Use the severity and category tables.
* **Never call something a false negative from this notebook alone.** The notebook has no ground truth. A
  behavior Cisco did not report is a *not reported* behavior until someone verifies it exists in the files and
  that a Cisco rule was expected to cover it.
* **Counts are per finding; skills are counted separately.** Cisco emits one finding per matching line, so one
  script with eight `requests.get` calls inflates finding counts. Skill-level columns are the fairer unit.
* **Analyzer labels differ from analyzer flags.** Findings carry `analyzer` values such as `static`,
  `correlation`, `behavioral`; `--use-behavioral` adds the `behavioral` label but the default analyzers still run.

### What Cisco (2.3.1, `balanced`) detects for RQ1-B

Well covered by deterministic rules: hardcoded secrets and credential-file reads; outbound network calls and
exfiltration-shaped POSTs/sockets; eval/exec, `os.system`, shell pipelines; prompt-injection phrases and Unicode
steganography; undeclared network use versus the manifest; unpinned/untrusted package installs; obfuscation and
embedded binaries. **Not covered as such:** general file-system modification (only path traversal and hidden-file
globbing), and account modification or other sensitive external actions. The LLM judge is the only Cisco component
that could reason about *purpose* ("this skill says it manages credentials, so reading secrets is expected");
it is optional, paid, and sends skill text to a provider, so it is a later, separately approved experiment.

### Known limitations of this setup

* **Context is bounded by GitSkills.** The sample stores text for about 57 % of supporting files; folders with
  binaries, oversized files or more than the capture limit are incomplete and their full-folder scans are
  skipped by default. Incomplete folders are a category of result, not missing data to be ignored.
* **Only representatives are scanned.** Copies (`dedup_primary = 0`) resolve to their representative; a copy
  in a different repository could in principle ship different supporting files, which GitSkills does not store.
* **Frontmatter.** Skills with invalid frontmatter (about 13 % of distinct skills) are scanned with Cisco's
  fallback (`SKILL_LOAD_FALLBACK_USED`, INFO). This is recorded, not hidden.
* **Scanner version and policy are fixed per run** and recorded in `run_metadata.json`; mixing runs from
  different versions or policies invalidates the finding-level comparisons.
* **Runtime.** Each scanner call takes about 5–10 s of wall time (mostly process start-up; the scan itself is
  under a second). Four configurations per skill ≈ 30 s per skill.

### Scaling to the 400-skill stratified sample

The per-skill machinery here (export → four scans → flatten) is already batch-shaped; what the 400-skill
pipeline needs on top of it:

1. **Sampling, not keyword search.** Define strata on indexed GitSkills columns of `dedup_primary = 1` rows —
   e.g. no supporting files / references only / scripts only / scripts and references — report each stratum's
   population size, draw with `ORDER BY random()` seeded through a Python RNG over the stratum's row IDs (SQLite's
   `RANDOM()` cannot be seeded), and save a `sample_manifest.csv` with the stratum, population size, and the
   selected `repo`, `path`, `file_sha`. Restrict to complete folders *or* sample them and report the
   incomplete share — decide and document, do not let completeness silently bias the sample.
2. **Deduplication** is already handled by `dedup_primary`; add a check for identical supporting-file sets
   (same set of `entry_sha` values) if near-duplicate folders turn out to be common.
3. **Resumability.** `load_report` plus the atomic save already make re-running safe; add a "skip if a VALID
   report exists" rule in Section 7 and a sample-wide `scan_summary.csv` that appends per batch instead of
   overwriting.
4. **Throughput.** 400 skills × 4 configurations ≈ 1,600 scanner calls ≈ 2.5–4.5 h serially; run batches in
   parallel processes or use Cisco's Python API in-process (same analyzers; verify identical output on the
   pilots first).
5. **Aggregation.** Point Sections 8–13 at the union of all `reports/*.json` under the run folder rather than
   at the in-memory `reports` of one batch, and add skill-level denominators per stratum.
6. **Manual review protocol.** Two reviewers, the `interpretation` vocabulary above, and inter-rater agreement
   on a subsample, before any claim about legitimate vs. problematic behavior.